<!-- colab-badge -->
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/exercises/Ex12.2-power-grid-stability-prediction/Ex12.2_gnn_contingency_screening.ipynb)

*Open this notebook in Google Colab. Its first code cell fetches the library files from the public course repository, so nothing needs uploading.*

<!-- course-header v3 -->

**Deep Learning for Engineering** · MSc, Aalborg University · 2026

Developed by **Remus Teodorescu** (ret@et.aau.dk), with support from Research Assistant **Noman Khan** (nomank@energy.aau.dk).

*Reference texts — read for the theory. Used as an **inspirational source** for this course, not as a source of its code:*

- Liu, *PINN with Python*, 2025.
- Raissi, Perdikaris & Karniadakis, *Physics-informed neural networks*, J. Comput. Phys. 378 (2019) 686–707.
- Prince, *Understanding Deep Learning*, MIT Press 2023.

Every notebook in this course has been **written and rewritten by the authors named above**. The code, the problems, the data and the exposition are **original to this course** and are not derived from any publisher's code listings or companion notebooks. Where notation matches a textbook's it is the standard notation of the field, and where an idea is a named author's it is cited as theirs in the text.

See `docs/PROVENANCE.md` for what each reference is cited for, set by set.

---

# Ex_12.2 — Screening Contingencies

**Deep Learning for Engineering · Aalborg University · Part 2 · paired with L12.2 · Prediction of Power Grid Stability**

A line faults. The protection needs 140 ms to clear it. Does the local plant
stay in step with the rest of the grid? The answer is the **critical clearing
time**: the longest fault the plant survives. Finding it takes a simulation in
time, for every line that could fault and every state the grid could be in.
You compute it two ways - by simulation, and by a graph network that has
learned from simulations - and compare them on accuracy and on time. Then you
ask the network about a line it has never seen out.

| | what you do | TODO |
|---|---|---|
| **1 – 3** | the problem, its data and its physics | — |
| **4** | the reference, and the simulation on a coarser step for an agreed 5 ms | — |
| **5** | a graph network for the critical clearing time, and the comparison | the graph layer, the loss |
| **6** | a topology the network has not seen, and the threshold of the screen | — |
| **7** | what the notebook says | — |
| **8 – 9** | your report, and the mini project proposal | your answers |

About a minute on a CPU. Numbers are printed with at most two decimals.

---

## 0 · Setup

In [ ]:
# files-cell v2 ----------------------------------------------------------
# The library files must sit beside the notebook. Locally they already do. On
# Google Colab they are fetched from the public course repository on every
# run, so a file left over from an earlier session cannot shadow it.
import os, sys, urllib.request
FILES = ['course_core.py', 'pinn_core.py', 'problem.py']
URL = "https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/exercises/Ex12.2-power-grid-stability-prediction/"
ON_COLAB = "google.colab" in sys.modules
for f in FILES:
    if ON_COLAB or not os.path.exists(f):
        urllib.request.urlretrieve(URL + f, f)
        print("fetched", f)
for _m in ("course_core", "pinn_core", "problem"):
    sys.modules.pop(_m, None)
print("files ready:", ", ".join(FILES))

In [ ]:
# --- setup ---------------------------------------------------------------
import os, time
for f in ("course_core.py", "pinn_core.py", "problem.py"):
    assert os.path.exists(f), f"{f} is missing - run the files cell above first"
from pinn_core import *                                  # noqa: F401,F403
import course_core as cc
import problem as pb
import numpy as np, torch, matplotlib.pyplot as plt
import scipy.sparse as sp, scipy.sparse.linalg as spla

cc.OUTPUT_DIR = "Ex12.2_outputs"                              # where the report is written
set_seed(88)  # fix the random start so a run repeats
print("torch", torch.__version__, " device:", DEVICE, " dtype:", torch.get_default_dtype())

def middle(fn, reps=5):                                    # the middle of five timings, after a warm-up
    fn(); ts = []
    for _ in range(reps):
        t0 = time.perf_counter(); fn(); ts.append(time.perf_counter() - t0)
    return float(np.median(ts))

---

## 1 · The problem

The grid of Ex_12.1: six buses, six lines, the Nordic system at bus 0 and a
local plant at bus 1. A **contingency** is a solid three-phase fault at the
sending end of a line, cleared by switching that line out. There are six: the
fault at the plant with no line lost, and five line outages. (The sixth line
is not screened: losing it cuts bus 5 off, which is loss of supply, not a
stability case.)

Wanted, for every operating point and every contingency: the critical clearing
time (CCT), and from it the verdict. **A case is insecure if its critical
clearing time is 140 ms or less**: the fault outlasts the breakers and the
plant falls out of step.

## 2 · The data

| | |
|---|---|
| operating points | 180: load from 0.85 to 1.60 of base, the plant's share of it, the HVDC import |
| contingencies | 6, so 1080 cases |
| per bus, 6 numbers | $P$, $Q$, $|V|$, $\theta$ before the fault; is a machine here; is the fault here |
| the graph | the lines left after the fault is cleared: one 6 × 6 adjacency per case |
| the label | the critical clearing time, searched up to 500 ms |
| split | 144 operating points to train on, 36 held out - whole operating points, never single cases |

**The grid is representative of eastern Denmark, not a model of it**, and the
plant's inertia (4 s), damping and reactance are estimates for a unit of its
size. The labels come from the classical machine model: no exciter, no
governor, first swing only.

## 3 · The physics

**The swing equation** (L12.2's *Transient Stability*). Each machine's rotor
angle $\delta_i$ accelerates with the difference between the mechanical power
driving it and the electrical power it delivers:

$$\frac{d\delta_i}{dt} = \omega_i - \omega_s, \qquad
  \frac{2H_i}{\omega_s}\frac{d\omega_i}{dt} = P_{m,i} - P_{e,i}(\delta) - D_i\,\frac{\omega_i - \omega_s}{\omega_s}$$

with $H_i$ the inertia constant, $D_i$ the damping and $\omega_s$ the
synchronous speed. During the fault the plant can deliver almost no power, so
it speeds up. When the fault is cleared it can deliver power again, on a grid
with one line fewer, and slows down - if it has not gone too far.

**The critical clearing time** is the longest fault for which the angle
between the two machines stays under 180°. The cell below prints the problem
and shows one case cleared 10 ms before its critical time and 10 ms after.

In [ ]:
pb.describe_problem()
cases = pb.contingencies()                                    # the six contingencies
base = np.stack(pb.injections())                              # the base operating point: P and Q at every bus
cct_one, one = pb.label_case(base, cases[1]["outage"], cases[1]["fault_bus"], return_detail=True)
print(f"  base operating point, line 0 out: critical clearing time {1000 * cct_one:.0f} ms")

fig, ax = plt.subplots(figsize=(8, 3.2))
for extra, style in ((-0.010, "-"), (0.010, "--")):
    T, D, W = pb.simulate_swing(one["machines"], one["Yred_pre"], one["Yred_post"], t_end=1.5, t_fault=pb.T_FAULT,
                                t_clear=pb.T_FAULT + cct_one + extra, Yred_fault=one["Yred_fault"], delta0=one["delta0"])
    ax.plot(T, np.degrees(D[:, 1] - D[:, 0]), style, lw=2, label=f"cleared after {1000 * (cct_one + extra):.0f} ms")
ax.axhline(180, color="0.5", lw=1); ax.set_ylim(0, 360)
ax.set_xlabel("time  [s]"); ax.set_ylabel("angle between the machines  [deg]"); ax.legend(frameon=False); plt.tight_layout(); plt.show()

**What you should see.** The six contingencies, and for line 0 out at the
base operating point a critical clearing time of 220 ms: secure, with 80 ms to
spare. Cleared 10 ms earlier, the angle swings out past 120°, turns round and
comes back. Cleared 10 ms later, it runs away. **Twenty milliseconds separate
a non-event from a plant out of step.**

---

## 4 · The reference, and the simulation on a coarser step

**The reference** is the simulation: the swing equation integrated by a
fourth-order Runge–Kutta method on a step of 2 ms, and the clearing time
bisected to 2 ms. Each case takes eight simulations. `pb.build_cases` does it
for all 1080 cases at once.

**The classical way to make it cheaper** is a coarser step. The fault can only
be cleared at a step, so the step sets how exactly the clearing time is known.

**The agreed accuracy is 5 ms** on the worst case: under 4 % of the
protection's 140 ms.

In [ ]:
data = pb.build_cases(180, seed=12)                           # the reference: step 2 ms, bisection to 2 ms
X, A, cct, op_id, cont_id = data["X"], data["A"], data["cct"], data["op_id"], data["cont_id"]
t_ref = float(data["seconds"]) / len(cct)
insecure = cct <= pb.PROTECTION_TIME
print(f"  reference: {len(cct)} cases, {1000 * t_ref:.2f} ms a case; {insecure.sum()} insecure ({100 * insecure.mean():.0f} %), "
      f"{(cct >= pb.CCT_MAX).sum()} at the 500 ms ceiling")
for c in cases:
    m = cont_id == c["index"]
    print(f"    {c['label'][:44]:44s} median {1000 * np.median(cct[m]):3.0f} ms, insecure in {100 * insecure[m].mean():2.0f} %")

AGREED = 0.005
coarse = {}
for step, tol in ((0.005, 0.004), (0.010, 0.008), (0.020, 0.016)):
    d = pb.build_cases(180, seed=12, dt=step, tol=tol)        # the same simulation, coarser
    err = np.abs(d["cct"] - cct)
    wrong = int(((d["cct"] <= pb.PROTECTION_TIME) != insecure).sum())
    coarse[step] = (float(err.mean()), float(err.max()), wrong, float(d["seconds"]) / len(cct), d["cct"])
    print(f"  step {1000 * step:2.0f} ms: mean error {1000 * err.mean():.2f} ms, worst {1000 * err.max():.2f} ms, "
          f"{wrong} verdicts changed, {1000 * coarse[step][3]:.2f} ms a case" + ("   <- meets 5 ms" if err.max() <= AGREED else ""))
STEP = 0.005                                                  # the coarsest step that meets it

**What you should see.** 1080 cases at under 2 ms each; 16 % of them
insecure, almost all with line 0 or line 1 out or the fault at the plant, and
17 % at the 500 ms ceiling. **A step of 5 ms is the coarsest that meets 5 ms**
(worst error 4.9 ms), at about a third of the cost. Even so it changes 19
verdicts: many cases sit within a few milliseconds of 140 ms, and for those a
small error in the clearing time is a different answer.

On a real grid a case is not a millisecond. It is thousands of buses and a
detailed machine model, and a screening run is every line times every hour.

---

## 5 · A graph network for the critical clearing time

**A contingency is a change to the graph** (L12.2's *The Physics-Informed
Graph Network*). A line switched out is an edge removed, not a number in a
list. The network is given the six numbers at every bus and the graph left
after the fault, and returns the critical clearing time.

**One graph layer** lets every bus combine its own numbers with its
neighbours':

$$H' = \tanh\!\big(H\,W_{\mathrm{self}} + b + \hat{A}\,H\,W_{\mathrm{neigh}}\big)$$

with $H$ the numbers at the six buses (one row per bus), $W_{\mathrm{self}}$
and $W_{\mathrm{neigh}}$ two weight matrices shared by all buses, $b$ a bias,
and $\hat{A}$ the adjacency with self-loops, scaled by the degrees of the
buses. One layer moves information one line along. The grid is three lines
across, so there are three layers; then the mean over the buses, and a small
dense network that returns one number.

**The loss** is the mean squared distance from the simulated critical
clearing time, on the 144 training operating points. **There is no physics in
the loss**: the physics is in the labels, and in the graph the network is
handed (L12.2's *Three Places to Put the Physics*).

**TODO 1** is the graph layer; **TODO 2** the loss.

In [ ]:
class GraphLayer(torch.nn.Module):                            # a network: layers in __init__, the computation in forward
    def __init__(self, n_in, n_out):
        super().__init__()
        self.own = torch.nn.Linear(n_in, n_out)               # dense layer on a bus's own numbers, with the bias
        self.neighbours = torch.nn.Linear(n_in, n_out, bias=False)   # dense layer on what the neighbours send
    def forward(self, A_hat, H):
        # TODO 1: the graph layer of section 5, before its tanh. A_hat has shape (N, 6, 6), H has shape (N, 6, features).
        #
        #   return self.own(H) + A_hat @ self.neighbours(H)       # own numbers, plus the neighbours' summed along the lines
        raise NotImplementedError("Write the graph layer")

In [ ]:
def loss_fn(model, A_t, H_t, y_t):
    # TODO 2: the loss - the mean squared distance between the network's output and the labels y_t.
    #
    #   return (model(A_t, H_t) - y_t).pow(2).mean()              # mean squared distance from the simulated clearing time
    raise NotImplementedError("Write the loss")

In [ ]:
class Screen(torch.nn.Module):                                # a network: layers in __init__, the computation in forward
    def __init__(self, hidden=32, depth=3):
        super().__init__()
        self.layers = torch.nn.ModuleList([GraphLayer(pb.N_FEATURE if k == 0 else hidden, hidden) for k in range(depth)])   # three graph layers
        self.head = MLP(n_in=hidden, n_out=1, n_hidden=32, n_layers=2)   # dense tanh network: the pooled numbers in, one number out
    def forward(self, A_hat, H):
        for k, layer in enumerate(self.layers):
            Z = torch.tanh(layer(A_hat, H))                   # one graph layer, then tanh
            H = H + Z if k > 0 else Z                         # a residual connection after the first layer
        return self.head(H.mean(dim=-2))                      # the mean over the six buses, then the dense head

A_hat = pb.normalised_adjacency(A)                            # self-loops added, scaled by the degrees
train = op_id < 144                                           # 144 operating points to train on, 36 held out

def fit(keep):
    # a network trained on the cases `keep`; returns it and a function from case indices to predicted CCT in seconds
    set_seed(88)                                              # fix the random start so a run repeats
    flat = X[keep].reshape(-1, pb.N_FEATURE)
    mean, spread = flat.mean(axis=0), flat.std(axis=0)
    Xs = (X - mean) / np.where(spread > 1e-12, spread, 1.0)   # every feature to zero mean and unit spread, on the training cases
    # the graphs and the bus numbers of the cases m, as tensors on the device
    tensors = lambda m: (to_tensor(A_hat[m]).reshape(-1, 6, 6).to(DEVICE), to_tensor(Xs[m]).reshape(-1, 6, pb.N_FEATURE).to(DEVICE))   # NumPy arrays -> tensors
    model = Screen().to(DEVICE)                               # the network, on the device
    A_t, H_t = tensors(keep)                                  # the training graphs and bus numbers
    y_t = to_tensor(cct[keep] / 0.1).to(DEVICE)               # the labels, in units of 100 ms
    t0 = time.perf_counter()
    # Adam first, then L-BFGS: the Part 2 training recipe
    train_two_stage(model, lambda: loss_fn(model, A_t, H_t, y_t), adam_steps=1500, lbfgs_steps=40, lr=3e-3, report_every=0)
    seconds = time.perf_counter() - t0
    def predict(m):
        with torch.no_grad():  # no gradient tracking: evaluation only
            return to_numpy(model(*tensors(m))).ravel() * 0.1  # back to seconds
    return model, predict, seconds

In [ ]:
net, predict, train_time = fit(train)
print(f"{parameter_count(net)} weights, {train.sum()} training cases, trained in {train_time:.0f} s")

In [ ]:
test = ~train                                                 # the 36 held-out operating points: 216 cases
pred = predict(test)
t_net = middle(lambda: predict(test)) / test.sum()            # all 216 at once

def row(name, p, truth, seconds, training="-"):
    c = pb.confusion(truth, p)                                # the verdicts at 140 ms: missed insecure cases, false alarms
    err = np.abs(p - truth)
    print(f"  {name:34s}{1000 * err.mean():8.2f} ms{1000 * err.max():8.2f} ms{c['missed']:6d} of {c['tp'] + c['fn']:2d}{c['alarms']:9d}{1e6 * seconds:10.0f} us{training:>10s}")
    return err.mean(), err.max(), c["missed"], c["alarms"]

print(f"{'216 held-out cases':36s}{'mean error':>11s}{'worst':>11s}{'missed':>12s}{'alarms':>9s}{'one case':>13s}{'training':>10s}")
print(f"  {'reference, step 2 ms':34s}{'-':>11s}{'-':>11s}{'-':>12s}{'-':>9s}{1e6 * t_ref:10.0f} us{'-':>10s}")
r_coarse = row("simulation, step 5 ms", coarse[STEP][4][test], cct[test], coarse[STEP][3])
r_net = row("graph network", pred, cct[test], t_net, f"{train_time:.0f} s")

fig, ax = plt.subplots(figsize=(5.2, 4.2))
ax.plot([0, 520], [0, 520], "k--", lw=1); ax.plot(1000 * cct[test], 1000 * pred, ".", ms=4)
ax.axvline(140, color="0.5", lw=1); ax.axhline(140, color="0.5", lw=1)
ax.set_xlabel("simulated critical clearing time  [ms]"); ax.set_ylabel("the network's  [ms]")
ax.set_title("held-out operating points", fontsize=9); plt.tight_layout()
plt.savefig(cc.output_path("Ex12.2_report.png"), dpi=150); plt.show()

**What you should see.** On operating points it has never seen, the network
is within about 0.6 ms on average and 4 ms at worst: **more exact than the
simulation on a 5 ms step**, with no insecure case missed and no false alarm
(the coarse simulation raises 5), at a few microseconds a case against half a
millisecond. Training took about
ten seconds, and the 864 labels it was trained on took the reference about a
second and a half.

Read that with care. The network is not solving the swing equation. It has
learned how the critical clearing time varies with load and import **on six
graphs it has seen hundreds of times each**.

---

## 6 · A topology the network has not seen, and the threshold of the screen

**The fair test** (L12.2's *How to Find Out Whether the Physics Helped*) is a
graph that was never in the training set. The cell trains the same network
again, on the same 144 operating points, **without any case that has line 0
out**, and then asks it about line 0 out on the 36 held-out operating points.
Line 0 is the direct tie between the plant and the Nordic system, and the
worst contingency of the six.

**The threshold of the screen.** A screen flags a case for simulation if the
predicted clearing time is under a threshold. At 140 ms it flags what it
believes insecure. Raise the threshold and it misses fewer insecure cases and
raises more false alarms (L12.2's *Screening, Not Replacement*). The cell
raises it until no insecure case is missed.

In [ ]:
line0 = cont_id == 1                                          # the cases with line 0 out
net_u, predict_u, _ = fit(train & ~line0)                     # trained without ever seeing line 0 out
held = test & line0                                           # line 0 out, on the held-out operating points
pred_u = predict_u(held)
print(f"{'line 0 out, 36 held-out cases':36s}{'mean error':>11s}{'worst':>11s}{'missed':>12s}{'alarms':>9s}")
for name, p in (("network that has seen line 0 out", predict(held)), ("network that has not", pred_u)):
    c = pb.confusion(cct[held], p); err = np.abs(p - cct[held])
    print(f"  {name:34s}{1000 * err.mean():8.2f} ms{1000 * err.max():8.2f} ms{c['missed']:6d} of {c['tp'] + c['fn']:2d}{c['alarms']:9d}")
bias = float(np.mean(pred_u - cct[held]))
print(f"  the network that has not seen it is {1000 * bias:+.0f} ms off on average: it says the grid is safer than it is")

thresholds = np.arange(0.140, 0.3001, 0.005)
bad = cct[held] <= pb.PROTECTION_TIME                         # the truly insecure cases
missed = [int((bad & ~(pred_u <= th)).sum()) for th in thresholds]
alarms = [int((~bad & (pred_u <= th)).sum()) for th in thresholds]
first = int(np.argmax(np.array(missed) == 0))                 # the lowest threshold that misses nothing
print(f"  threshold raised from 140 ms to {1000 * thresholds[first]:.0f} ms: no insecure case missed, {alarms[first]} false alarms among {int((~bad).sum())} secure cases")

**What you should see.** The network that has seen line 0 out is within a
millisecond on it. **The network that has not is about 57 ms off, always on
the same side**: it predicts the clearing time of a grid that still has its
strongest line, and so **it misses all 11 insecure cases**.

With the threshold raised from 140 ms to 185 ms it misses none and raises no
false alarm. The network is off by a nearly constant amount, so it still puts
the cases in the right order. But 185 ms was found with the true answers in
hand. For a line that has never been simulated, nobody knows how far to raise
it.

---

## 7 · What the notebook says

* **The simulation is the truth here, and it is what costs.** Eight
  integrations of the swing equation per case, for every line and every
  operating point. A coarser step is the classical saving, and at 5 ms it
  already changes verdicts near the limit.
* **On graphs it has seen, the network was faster and more exact than the
  coarse simulation.** That is interpolation over load and import, on six
  graphs, and for that it is very good.
* **On a graph it had not seen, it was wrong, and wrong on the unsafe side.**
  A graph network can be asked about a new topology, which a dense network
  cannot. Being able to ask is not the same as getting the right answer.
* **It still ranked the unseen cases correctly.** Its error was a nearly
  constant offset, so a raised threshold separated them. The right threshold
  was only known because the truth was.
* **So it screens; it does not decide.** The cases it is unsure of, and every
  topology outside its training, go to the simulation.
* **There is no physics in this network's loss.** The physics is in the labels
  and in the graph. L12.2's physics-informed graph network, with the swing
  equation as a residual, is not built here; whether it does better on the
  unseen line is the experiment the lecture describes.
* **A margin says more than a verdict.** The clearing time minus 140 ms tells
  an operator how close a case is. Cases a few milliseconds from the limit are
  the ones a small error flips.
* **The labels are the classical machine model's**: no exciter, no governor,
  first swing only, on a grid whose impedances are textbook values. The
  supplement compares them with a fuller model in ANDES.

---

## 8 · Your report

**What these cells do.** You answer the questions below; the next cell writes `Ex12.2_report.md` with the results of section 5 and 6 and your answers, and the cell after it turns the report into a PDF for Moodle.

Each question builds part of an answer to one of L12.2's questions for the oral examination; the arrow says which. Keep each answer to a few sentences, with your numbers. Write your name in `NAME`, and each answer between the triple quotes that follow its question; an answer needs at least eight words.

In [ ]:
# TODO: write your answers. Every string below must be replaced.

NAME = "Your Name"

ANSWERS = {
    "Why is screening combinatorial, and what did one case cost the reference in this notebook? Say what that cost becomes on a grid of thousands of buses, as far as the notebook lets you. (-> L12.2 Q1)": """
""",
    "Section 3 showed one fault cleared 10 ms before its critical time and 10 ms after. Explain the two curves with the equal area criterion, and say why the clearing time minus 140 ms is worth more to an operator than the verdict. (-> L12.2 Q2)": """
""",
    "A line switched out is an edge removed. What did the graph network do with that, in your own TODO 1, that a dense network on a list of 36 numbers could not? (-> L12.2 Q3)": """
""",
    "Compare the network on held-out operating points with the network on a held-out topology. Which of the two tests would you report, what did it take to make the screen miss nothing, and could you have known that threshold without the simulation? (-> L12.2 Q4)": """
""",
    "To conclude: would you put this network in front of an operator, and for what? Say what it may decide, what it must pass on to the simulation, and what you would do before trusting it on a line it has not seen. (-> L12.2, Exercise slide)": """
""",
}

In [ ]:
MIN_WORDS = 8                                               # an answer is a few sentences, not a word
short = [(i, len(text.split())) for i, text in enumerate(ANSWERS.values(), 1) if len(text.split()) < MIN_WORDS]
if NAME.strip() in ("", "Your Name") or short:
    print("Not ready - the report is written once these are fixed:")
    if NAME.strip() in ("", "Your Name"):
        print("  NAME: write your name")
    for i, n in short:
        print(f"  answer {i}: {n} word(s); write at least {MIN_WORDS}")
else:
    rows = ["| 216 held-out cases | mean error | worst | insecure missed | false alarms | one case | training |", "|---|---|---|---|---|---|---|",
            f"| reference, step 2 ms | - | - | - | - | {1e6 * t_ref:.0f} us | - |",
            f"| simulation, step 5 ms | {1000 * r_coarse[0]:.2f} ms | {1000 * r_coarse[1]:.2f} ms | {r_coarse[2]} | {r_coarse[3]} | {1e6 * coarse[STEP][3]:.0f} us | - |",
            f"| graph network | {1000 * r_net[0]:.2f} ms | {1000 * r_net[1]:.2f} ms | {r_net[2]} | {r_net[3]} | {1e6 * t_net:.0f} us | {train_time:.0f} s |",
            "", f"Line 0 out, never seen in training: {1000 * np.abs(pred_u - cct[held]).mean():.2f} ms off on average, bias {1000 * bias:+.0f} ms, "
            f"{pb.confusion(cct[held], pred_u)['missed']} of {int(bad.sum())} insecure cases missed",
            "", f"Threshold for no missed case: {1000 * thresholds[first]:.0f} ms, with {alarms[first]} false alarms among {int((~bad).sum())} secure cases",
            "", "The grid's impedances are textbook values; the labels are the classical machine model's."]
    out = ["# Ex_12.2 — Screening Contingencies", "", f"**{NAME}**", "", "Deep Learning for Engineering · Aalborg University · 2026", "",
           "## Results", "", *rows, ""]
    for qq, text in ANSWERS.items():
        out += [f"## {qq}", "", text.strip(), ""]
    path = cc.output_path("Ex12.2_report.md")
    with open(path, "w", encoding="utf-8") as fh:
        fh.write("\n".join(out))
    print("written", path)

**What you should see.** `Not ready` until every answer is yours; then the path of the report. Then run the cell below: it turns the report into a PDF, with the figure of section 5, and downloads it. **Upload the PDF.**

In [ ]:
# Report as PDF for Moodle -----------------------------------------------
# Runs after the report cell above: turns Ex12.2_report.md into Ex12.2_report.pdf, with any figure
# saved as Ex12.2_report*.png embedded above the answers, and downloads it. Upload
# the PDF to Moodle; the .md stays as the source.
import subprocess, sys, glob, os
pdf_path = os.path.join(cc.OUTPUT_DIR, "Ex12.2_report.pdf")
try:
    import markdown, weasyprint
except ImportError:                       # installed already on a second run
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "markdown", "weasyprint"])
    import markdown, weasyprint

md_path = os.path.join(cc.OUTPUT_DIR, "Ex12.2_report.md")
if not os.path.exists(md_path):
    print("No report yet: it is written once every answer above is filled in.\n"
          "Fill them in, run the cell that writes the report, then run this one again.")
else:
    md = open(md_path, encoding="utf-8").read()
    figs = sorted(glob.glob("Ex12.2_report*.png")
                  + glob.glob(os.path.join(cc.OUTPUT_DIR, "Ex12.2_report*.png")))
    if figs:
        imgs = "\n\n".join(f"![{os.path.basename(p)}]({p})" for p in figs)
        i = md.find("\n## ", md.find("## Results") + 1) if "## Results" in md else -1
        md = (md[:i] + "\n\n" + imgs + "\n" + md[i:]) if i > 0 else md + "\n\n" + imgs + "\n"

    html = markdown.markdown(md, extensions=["fenced_code", "tables"])
    css = """body{font-family:Helvetica,Arial,sans-serif;font-size:11pt;margin:2cm}
    h1{font-size:18pt} h2{font-size:13pt;margin-top:18pt}
    pre{background:#f3f4f6;padding:8px;font-size:9.5pt} img{max-width:100%}"""
    weasyprint.HTML(string=f"<html><head><meta charset='utf-8'><style>{css}</style></head>"
                           f"<body>{html}</body></html>", base_url=".").write_pdf(pdf_path)
    print("written", pdf_path, f"with {len(figs)} figure(s)" if figs else "")
    try:
        from google.colab import files
        files.download(pdf_path)
    except ImportError:
        pass

---

## 9 · Mini project proposal

Two problems grown out of this set, for you to take further on your own. Each
student chooses **one** mini project from the Part 2 sets and has **one month**
to solve it individually. The dates and the hand-in are given in L13.

The ground truth is **given to you**, built by the course with a classical
solver and shown to have converged; the worked example below each project is
its first case. You do not solve the physics a second way: the month goes into
the deep learning - making a network predict what the ground truth says, as
accurately and as fast as the requirements ask.

### MP12.2A · A bigger grid, and conditions it never saw

**The problem.** The set screened single-line outages on one operating point. A control room's
screener must also be right at loads it was never trained on - on a hot
afternoon or a quiet night. Here the grid is the WSCC 9-bus system and the
question is how the screener does outside its training range.

| | |
|---|---|
| grid | the WSCC 9-bus system of MP12.1A |
| cases | each of the six transmission lines opened by a three-phase fault at either of its ends - 12 cases |
| load | nine levels, 70 to 130 % of the book's loads and generation |
| training | 80 to 120 %; testing at 70, 75, 125 and 130 % |

Double outages are not used: the six lines form one ring, so any two out split
the grid.

**What you build.** The graph network of this set, trained to predict the critical clearing time
from the operating point and the topology, on the training loads; then tested
outside them.

**The ground truth you get.** `MP12.2A_truth.npz`, from `tools/miniprojects/ex122_truth.py` on
`grid9.py`: the critical clearing time of all 108 cases by bisection on the
time-domain simulation, to 1 ms (1.31 s per case, 141 s for all). A value of
0.600 s means stable up to the 0.6 s the search allows.

**Worked example.** The critical clearing time falls from 0.279 s at 70 % load to 0.104 s at 130 %.
The worst case changes with the load: line 7-8 with the fault at bus 7 up to
80 %, line 5-7 from 90 %. Cleared at 0.15 s, 3 % of the training cases and 10 %
of the held-out ones are unstable - the held-out loads are where it matters.

**Requirements.**

* The critical clearing time within **10 ms** on the training loads, and within **20 ms** on the held-out ones.
* Every case unstable at 0.15 s flagged, on the held-out loads too.
* All 108 cases in under **0.1 s**, against 141 s for the simulation.

**What you hand in.** Accuracy on the training and on the held-out loads, the dangerous cases your
screener missed, and whether you would let it replace simulation at 130 %.

### MP12.2B · Which line tripped?

**The problem.** The set asked what happens if a line is lost. The inverse is asked every day:
a line has opened, the meters recorded the grid's swing, and nobody yet knows
which line it was. From the records alone, find the line.

| | |
|---|---|
| grid | the WSCC 9-bus system of MP12.1A |
| events | each of the six lines opens, with no fault, at five load levels from 80 to 120 % - 30 events |
| records | the three machines' frequencies every 10 ms for 2 s, noise 1 mHz |

**Why a plain PINN is not enough.** Two lines in similar places give similar records, and the load level changes a
record as much as the line does. Which outages can be told apart with the
meters there are is part of the answer.

**What you build.** A graph network that takes the records and the grid's topology and returns the
line that tripped, with a confidence. You show which pairs of lines it
confuses, and why.

**The ground truth you get.** `MP12.2B_truth.npz`, from the same script: the 30 events' records, clean
and with noise, and the line and load of each.

**Worked example.** The closest pair is lines 4-5 and 4-6: over any two load levels their records
come within 21.50 mHz of each other - twenty times the noise, but a hundredth of
the most distinct pair (lines 5-7 and 8-9, 998.6 mHz). The next closest are 4-6
and 8-9 at 116.82 mHz.

**Requirements.**

* Every one of the 30 events identified, with noise, and a confidence for each.
* At 5 mHz noise, say which pairs you still separate.
* The line that tripped in under **10 ms** - inside a control room's cycle.

**What you hand in.** The share of events identified, the pairs your network confuses and at what
noise, and one meter you would add to separate lines 4-5 and 4-6.